# Program (b): hand-written flow-control program

Target T2 (first failures), project `atlasdb`, N = 10 (the number of selected features during execution of `feature_selection.ipynb`).

The program is distilled from the dev-segment GP runs (`gp.ipynb`), not searched again. GP in this form is essentially a rule generator over the domain of functions that program (b) needs.

Protocol:
1. Skeletons are written from the best dev GP program and the most frequent GP motifs.
2. The dev check (dev fit / dev val) picks one skeleton and freezes it.
3. On each CV fold, only the threshold values are re-tuned: coordinate descent on MCC over the 5%–95% quantiles of the fold's fit rows. The tuned values or the dev defaults are kept, whichever scores higher on val.
4. The test fold is scored once, with the same metrics as (a).

Caveat: (b) is derived from (a)'s programs, so it is not independent of (a). The comparison measures how much performance survives when a person simplifies the program, not human intuition against evolution.

In [ ]:
import inspect
import itertools
import pathlib
from functools import partial

import numpy as np
import pandas as pd
import plotly.io as pio
from deap import gp
from IPython.display import Markdown, display

from acise_p1_cigp.analysis.features import load_features
from acise_p1_cigp.data.load import load_project, preprocess
from acise_p1_cigp.gp.engine import make_pset
from acise_p1_cigp.gp.engine import predict as gp_predict
from acise_p1_cigp.metrics.metrics import scores
from acise_p1_cigp.solutions.flow_control.model import CANDIDATES, THRESHOLDS, predict
from acise_p1_cigp.solutions.flow_control.tune import count_statements, fit_predict, run
from acise_p1_cigp.stats.tests import paired_wilcoxon, per_fold, position_per_fold, summary, vargha_delaney_a12
from acise_p1_cigp.training.cv import dev_split, folds, prepare, run_cv
from acise_p1_cigp.utils.logging.log import get_logger, set_level
from acise_p1_cigp.viz import plots

pio.renderers.default = "png"  # static only; notebook+png embeds plotly.js (5 MB)
logger = get_logger("notebook.handwritten")

TARGET, PROJECT, METHOD = "T2", "atlasdb", "flow_control"
GP_OUT = pathlib.Path("../results/gp").resolve()
OUT = pathlib.Path("../results/flow_control").resolve()
OUT.mkdir(parents=True, exist_ok=True)


def show(src: str) -> None:
    display(Markdown(f"```python\n{src}```"))

## Project loading...

In [ ]:
ds = preprocess(load_project(PROJECT), TARGET)
features = load_features(TARGET, PROJECT)
gp_cv = pd.read_csv(GP_OUT / f"cv_{TARGET}_{PROJECT}.csv")
logger.info(f"Loaded {len(ds.X)} builds, {int(ds.y.sum())} failures, N = {len(features)}: {features}")

## Distillation source

Best dev GP program, the top hall-of-fame candidates, and the (feature, op) motifs across all dev runs. The structure is clear:
- `time_interval >= x` dominates the dev hall of fame, with about 515 uses;
- next come `has_old_change` low (about 96), then `pr_duration` high (about 80);
- `config_files` low, `src_churn` and `fail_ratio_pr_inc` follow far behind.

Thresholds here are in scaled units (percentiles); the dev program is printed in raw units.

In [ ]:
cand = pd.read_csv(GP_OUT / f"dev_candidates_{TARGET}_{PROJECT}.csv" )
show(cand.iloc[0]["python"])

display(cand[["program", "statements", "fitness", "val_fitness", "seed"]].head(10))
display(pd.read_csv(GP_OUT / f"motifs_{TARGET}_{PROJECT}.csv").head(10))

## Rules and their reasoning

Fail rate on dev rows, split at each rule's dev threshold (raw units). Base fail rate on dev: 145/427 = 34%.

| Rule | Dev split | Reasoning |
|---|---|---|
| `has_old_change >= 1` -> pass | 25% vs 41% | Compares the commit SHA lists (Git log) of the current change and the latest change. Likely reading (to confirm against RavenBuild): a positive value means this build re-includes commits that were already built, and on T2 that previous build passed, so part of the change is already verified. The GP puts this condition at the root of its best program. |
| `time_interval >= gap` -> fail | 43% vs 24% | A long gap since the previous build: more unbuilt upstream and dependency changes have piled up. |
| `pr_duration >= slow` -> fail | 44% vs 26% | The previous build was slow, i.e. it ran the full suite or a heavy part of the codebase. `pr_duration` is bimodal on dev (about 280 s vs. 1000 s and above). |
| `config_files < cfg` (S3 only, dropped) | 39% vs 26% for `config_files >= 1` | Matches the GP's `lt` motif, but the direction (more config changes, fewer failures) has no clear CI explanation; left to the dev check, which drops it. |


In [ ]:
d = ds.X.loc[ds.dev_mask, features].assign(failed=ds.y[ds.dev_mask])
for col, cut in ("has_old_change", 1), ("time_interval", 425), ("pr_duration", 1144.31), ("config_files", 1):
    display(d.groupby(d[col] >= cut)["failed"].agg(builds="size", fail_rate="mean").rename_axis(f"{col} >= {cut}"))

## Candidate skeletons

- S1: the best dev GP program as is.
- S2: S1 plus the slow-previous-build rule.
- S3: S2, with the long-gap rule restricted to builds with few config-file changes.

All are within the budget of N = 10. Two counts are given:
- `if`: the number of if/elif statements, the assignment's rule;
- `nodes`: the GP engine's count (if + comparisons + and/or/not), comparable with GP program sizes (median 4 on CV).

In [ ]:
for name, (p, th) in CANDIDATES.items():
    display(Markdown(f"**{name}**: {count_statements(p)}, thresholds {th}"))
    show(inspect.getsource(p))

In [ ]:
dev = dev_split(ds, features)
rows = []
for name, (p, th) in CANDIDATES.items():
    r = fit_predict(dev, p, th)
    rows.append({"skeleton": name, **count_statements(p), **r.payload["thresholds"],
                 "fit_mcc": r.payload["fit_mcc"], **scores(dev.y_val, r.pred)})
# S1 at the GP's own raw threshold (best dev program), not re-tuned
val_rows = dev.prep[0].transform(dev.raw_val[features]).to_dict("records")
p1 = CANDIDATES["S1"][0]
rows.append({"skeleton": "S1 @ GP thr", **count_statements(p1), "gap": 333.874,
             **scores(dev.y_val, run(p1, val_rows, {"gap": 333.874}))})
dev_check = pd.DataFrame(rows).set_index("skeleton")
dev_check.to_csv(OUT / f"dev_check_{TARGET}_{PROJECT}.csv")
display(dev_check[["if", "nodes", "gap", "slow", "cfg", "fit_mcc", "mcc", "precision", "recall"]].round(3))

**Frozen: S2.** Dev-val MCC is 0.208 for S2, 0.156 for S1 and 0.146 for S3. The best dev GP program reaches 0.249, and that is val fitness, i.e. after the size penalty.

S3's extra config rule loses recall on dev val (0.29 vs. 0.42 for S2) and is dropped. The structure is fixed from here on; the dev-tuned values become the defaults in `model.py`.

S1 matches the best dev GP program in structure only. Re-tuning on dev fit moves its threshold from 334 s to 108 s; at the GP's own 334 s (row `S1 @ GP thr`) S1 scores 0.254, above S2's 0.208 on dev val. That number is optimistic: the GP picked the program on these same 85 val rows. So the dev check compares skeletons under one tuner, not skeletons against the GP.


In [ ]:
show(inspect.getsource(predict))
display(Markdown(f"Defaults (dev fit): `{THRESHOLDS}`; statements: `{count_statements(predict)}`"))

## Cross-validation

The same 5 expanding-window folds as (a). (b) is deterministic, so there is one run per fold. GP rows (10 seeds per fold), Parrot and Majority are read from the GP CV results.

In [ ]:
set_level("WARNING")
fc, fc_pay = run_cv(ds, features, partial(fit_predict, predict=predict, thresholds=THRESHOLDS), METHOD)
set_level("INFO")
fc.to_csv(OUT / f"cv_{TARGET}_{PROJECT}.csv", index=False)
results = pd.concat([gp_cv, fc], ignore_index=True)
s = summary(results)
display(
    s[s.metric.isin(["mcc", "f1", "precision", "recall", "accuracy", "pct_builds_saved", "pct_failures_caught"])]
    .pivot(index="method", columns="metric", values=["mean", "std"])
    .round(3)
)

### Thresholds per fold

`source` says whether the tuned values or the dev defaults won on val.
- `gap` stays between 277 and 425 s across folds.
- In fold 3, `slow` rises to about 5279 s, the 95% quantile of that fold's fit rows: the rule narrows to the slowest 5% of previous builds (8% of test). Previous builds get slower over time (test median `pr_duration` is 282 s in fold 0 and 430–690 s later), so a fixed raw threshold would drift; per-fold tuning absorbs that.


In [ ]:
thr = pd.DataFrame([{"fold": k, **p["thresholds"], "source": p["source"], "fit_mcc": p["fit_mcc"],
                     "val_mcc": p["val_mcc"]} for (k, _), p in fc_pay.items()])
thr.to_csv(OUT / f"thresholds_{TARGET}_{PROJECT}.csv", index=False)
display(thr.round(3))
display(pd.DataFrame({"test_median_pr_duration": [ds.X["pr_duration"].iloc[f.test].median() for f in folds(ds)],
                      "test_share_slow": [(ds.X["pr_duration"].iloc[f.test] >= thr.loc[f.k, "slow"]).mean() for f in folds(ds)]}).round(3))


## (b) vs (a)

- Paired per-fold MCC: (b) against the GP's median over its 10 seeds.
- With 5 folds, the smallest two-sided Wilcoxon p is 0.0625 (`min_p`), so p < 0.05 is impossible. "Matches (a)" therefore rests on A₁₂ ≈ 0.5 and on where (b) falls within the GP's seeds in each fold (`position_per_fold`: 1.0 means (b) beats every seed).

In [ ]:
pf = per_fold(results, "mcc")
display(pf.round(3))
display(pf[[METHOD, "gp_so", "gp_mo"]].agg(["mean", "std"]).round(3))  # spread of fold medians
rows = []
for other in ("gp_so", "gp_mo"):
    w = paired_wilcoxon(pf[METHOD], pf[other])
    rows.append({"vs": other, "wilcoxon_p": w.p, "min_p": w.min_p, "n": w.n, "flag": w.flag,
                 "a12": vargha_delaney_a12(pf[METHOD], pf[other]), "b_wins": int((pf[METHOD] > pf[other]).sum())})
display(pd.DataFrame(rows).round(4))
display(pd.concat([position_per_fold(results, "mcc", METHOD, g) for g in ("gp_so", "gp_mo")], axis=1).round(3))

Result:
- Mean MCC: 0.189 for (b), 0.174 for GP SO, 0.170 for GP MO.
- Against SO: (b) wins 3 of 5 folds, A₁₂ = 0.56, p = 0.625.
- Against MO: (b) wins 2 of 5 folds, A₁₂ = 0.52, p = 1.0.
- Per fold, against the SO seeds: (b) beats 90% of them in fold 1, sits mid-distribution in folds 0 and 3, and falls in the bottom 10% in folds 2 and 4 (A₁₂ = 0.1). "Matches" holds overall, not fold by fold.
- Spread of the fold scores: std 0.059 for (b) vs. 0.099 for the GP SO fold medians; GP MO is as steady as (b) (0.060).
- Domain metrics: (b) and GP SO sit at the same point, about 65% of builds saved and 50% of failures caught. MO catches more failures (60%) but saves fewer builds (55%).

No difference can be detected: (b) matches (a) at 2 if-statements against a budget of N = 10.


In [ ]:
long = results.melt(id_vars=["method", "fold"], value_vars=["precision", "recall", "f1", "mcc", "accuracy"], var_name="metric")
plots.save(plots.metrics_box(long), f"fc_metrics_box_{TARGET}_{PROJECT}").show()

## Confusion matrix (b), summed over test folds

Accuracy misleads here: Majority (all-pass) reaches 0.753 accuracy with zero failures caught.

In [ ]:
c = fc[["tn", "fp", "fn", "tp"]].sum()
plots.save(plots.confusion(np.array([[c.tn, c.fp], [c.fn, c.tp]])), f"fc_confusion_{TARGET}_{PROJECT}").show()

## Fidelity to the GP

Share of test builds where (b) and each GP run's chosen program give the same prediction. GP programs are rebuilt from the programs CSV.

In [ ]:
programs = pd.read_csv(GP_OUT / f"programs_{TARGET}_{PROJECT}.csv")
pset = make_pset(features)
rows, base = [], []
for f in folds(ds):
    split = prepare(ds, features, f)
    b = fit_predict(split, predict, THRESHOLDS).pred
    preds = {}
    for _, r in programs[programs.fold == f.k].iterrows():
        a = gp_predict(gp.PrimitiveTree.from_string(r["program"], pset), pset, split.X_test)
        preds.setdefault(r["variant"], []).append(a)
        rows.append({"fold": f.k, "variant": r["variant"], "seed": r["seed"], "agreement": (a == b).mean()})
    for v, ps in preds.items():  # reference: how much two GP runs agree with each other
        base.append({"fold": f.k, "variant": v,
                     "agreement": np.median([(x == y).mean() for x, y in itertools.combinations(ps, 2)])})
fidelity = pd.DataFrame(rows)
fidelity.to_csv(OUT / f"fidelity_{TARGET}_{PROJECT}.csv", index=False)
display(pd.concat({
    "(b) vs GP": fidelity.groupby(["variant", "fold"])["agreement"].median().unstack(),
    "GP seed vs seed": pd.DataFrame(base).pivot(index="variant", columns="fold", values="agreement"),
}).round(3))


Median agreement of (b) with GP SO is 0.82–0.94 per fold (MO 0.67–0.97). Two GP SO runs agree with each other on 0.80–0.99 of test builds (MO 0.67–0.94): (b) is about as close to the GP as GP runs are to each other, with a fixed, readable structure.

### Appendix: all skeletons on CV

Reported for transparency only; **not** used for selection, since the freeze was decided on dev. S1 scores slightly higher on CV (mean 0.203 vs. 0.189). Switching to it now would be choosing on test data.

In [ ]:
set_level("WARNING")
app = pd.concat([run_cv(ds, features, partial(fit_predict, predict=p, thresholds=th), n)[0]
                 for n, (p, th) in CANDIDATES.items()], ignore_index=True)
set_level("INFO")
display(per_fold(app, "mcc").assign(**{"gp_so (median)": pf["gp_so"]}).round(3))